In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import seaborn as sns

from datetime import datetime, timedelta

from rctoolbox.utils.project_root import find_project_root

project_root = find_project_root()
os.chdir(project_root)
# print(f"Current working directory: {os.getcwd()}")

# Import custom modules
from rctoolbox.data.time_windows_generator import generate_time_windows
from rctoolbox.data.numpydataloader import DataLoader as NumpyDataLoader

In [ ]:
HEMS_ID = "AIBF"
OUT_DIR_PATH = project_root / "examples" / "run_identification" / "five_r1c_eniso13790" / HEMS_ID

## DATA

In [ ]:
DATA_DIR = project_root / "examples" / "data"

import pickle
with open(DATA_DIR / f"{HEMS_ID}_data.pkl", "rb") as f:
    DATA = pickle.load(f)

DATA["w2425"]["df"].columns
dv = "w2425"
df = DATA[dv]["df"]
df.index.min(), df.index.max()

print("season:        ", dv)
print("df shape:      ", df.shape)
print("time range:    ", df.index.min(), "->", df.index.max())

### TRAIN-VAL-TEST SPLITS

In [ ]:
wdh = 24; spb = int(1e10); ssh=1;
pre_window_lag = timedelta(hours=24)

t1_train = datetime(2024,12,1)
t2_train = datetime(2024,12,22)
tws_train = generate_time_windows(start_time=t1_train, end_time=t2_train, window_duration_hours=wdh, samples_per_batch=spb, slide_step_hours=ssh)

t1_val = datetime(2024,12,22)
t2_val = datetime(2025,1,1)
tws_val = generate_time_windows(start_time=t1_val, end_time=t2_val, window_duration_hours=wdh, samples_per_batch=spb, slide_step_hours=ssh)

tws = {"train": tws_train, "val": tws_val}
dataloader = NumpyDataLoader(df)
dataloader.load_all_data_batches(time_windows=tws, features=df.columns.tolist(), pre_window_lag=pre_window_lag)

In [ ]:
#  PREPARING TRAIN AND VAL BATCHES

from rctoolbox.models.rc.five_r1c_eniso13790.build_data_dict import build_data_dict
import joblib

data_joblib = {
    "train": build_data_dict(dataloader, "train"),
    "val": build_data_dict(dataloader, "val"),
}
data_joblib_path = OUT_DIR_PATH / "data_train.joblib"
joblib.dump(data_joblib, data_joblib_path)
# print("saved:", data_joblib_path)

In [ ]:
metric = "MAE"
model_config        = {"model_type": "5R1C_heat_demand", "init_config": {"method": "regime_aware", "variant": "last"}}
optimization_config = {"objective_function": metric, "std_weight": 0.0, "pop_size": 100, "n_gen": 150}
selection_config    = {"criteria": "val error", "select_R_star":True, "threshold_config": {"type": "idx", "x": 12000}}
clustering_config   = {"min_samples_ratio": 0.1, "cl_eval_metric": metric, "cl_eval_criteria": "val error"}

identification_config = {"model_config": model_config, 
                         "optimization_config": optimization_config, 
                         "selection_config": selection_config, 
                         "clustering_config": clustering_config

                        }
import json
with open(OUT_DIR_PATH / "identification_config.json", "w") as f:
    json.dump(identification_config, f, indent=4)